To Do:

- x make a version that has the grid iteration outside of the shiftgen method
    - x implement functioning version
    - x check if they produce the same results
- x compare both -> time measurement


In [1]:
from ponds import PONDS, GAUSS, SIGMOID

In [2]:
lat_size = 10
lon_size = 10
time_size = 100

pd = PONDS(
    lat_size=lat_size,
    lon_size=lon_size,
    time_size=time_size,
    background_noise=0.01,
    background_trend=0.00,
)

# 2) add random shift
pd.add_shift(
    random_seed=1,
    shape_method=GAUSS(random_seed=1),
    shift_method=SIGMOID(random_seed=1),
)

## Compare Computation Time

In [1]:
from ponds.blobgen_old import GAUSS as GAUSS_old
from ponds.blobgen import GAUSS as GAUSS_new

from ponds.core_old import PONDS as PONDS_old
from ponds.core import PONDS as PONDS_new

from ponds.shiftgen_old import SIGMOID as SIGMOID_old
from ponds.shiftgen import SIGMOID as SIGMOID_new

In [2]:
lat_size = 200
lon_size = 400
time_size = 500

# Number of times to run the entire script
n_runs = 10

In [4]:
import time
import numpy as np

# Store timings for each function call per run
time_init = []
time_shift1 = []
time_shift2 = []
time_total = []

for i in range(n_runs):
    print(f"Run {i+1}/{n_runs}", end="\r")
    start_total = time.time()
    
    # 1) initialize PONDS object
    t0 = time.time()
    pd_main = PONDS_old(
        lat_size=lat_size,
        lon_size=lon_size,
        time_size=time_size,
        background_noise=0.01,
        background_trend=0.002,
    )
    t1 = time.time()
    time_init.append(t1 - t0)
    
    # 2) add random shift
    t0 = time.time()
    pd_main.add_shift(
        random_seed=1,
        shape_method=GAUSS_old,
        shift_method=SIGMOID_old,
    )
    t1 = time.time()
    time_shift1.append(t1 - t0)
    
    # 3) add shift with custom parameters
    t0 = time.time()
    shift_dict = {
        "shift_steepness": 0.2,
        "shift_magnitude": 0.5,
    }
    pd_main.add_shift(
        random_seed=4,
        shape_method=GAUSS_old,
        shift_method=SIGMOID_old,
        kwargs=shift_dict,
    )
    t1 = time.time()
    time_shift2.append(t1 - t0)
    
    end_total = time.time()
    time_total.append(end_total - start_total)

# Compute averages
avg_old_init = np.mean(time_init)
avg_old_shift1 = np.mean(time_shift1)
avg_old_shift2 = np.mean(time_shift2)
avg_old_total = np.mean(time_total)

# Print results
print(f"Average initialization time: {avg_old_init:.4f} s")
print(f"Average first shift time: {avg_old_shift1:.4f} s")
print(f"Average second shift time: {avg_old_shift2:.4f} s")
print(f"Average total time per run: {avg_old_total:.4f} s")


Average initialization time: 2.4324 s
Average first shift time: 0.8999 s
Average second shift time: 2.2753 s
Average total time per run: 5.6076 s


In [5]:
import time
import numpy as np

# Store timings for each function call per run
time_init = []
time_shift1 = []
time_shift2 = []
time_total = []

for i in range(n_runs):
    print(f"Run {i+1}/{n_runs}", end="\r")
    start_total = time.time()
    
    # 1) initialize PONDS object
    t0 = time.time()
    pd_main = PONDS_new(
        lat_size=lat_size,
        lon_size=lon_size,
        time_size=time_size,
        background_noise=0.01,
        background_trend=0.002,
    )
    t1 = time.time()
    time_init.append(t1 - t0)
    
    # 2) add random shift
    t0 = time.time()
    pd_main.add_shift(
        random_seed=1,
        shape_method=GAUSS_new(),
        shift_method=SIGMOID_new(),
    )
    t1 = time.time()
    time_shift1.append(t1 - t0)
    
    # 3) add shift with custom parameters
    t0 = time.time()
    shift_dict = {
        "shift_steepness": 0.2,
        "shift_magnitude": 0.5,
    }
    pd_main.add_shift(
        random_seed=4,
        shape_method=GAUSS_new(),
        shift_method=SIGMOID_new(**shift_dict),
    )
    t1 = time.time()
    time_shift2.append(t1 - t0)
    
    end_total = time.time()
    time_total.append(end_total - start_total)

# Compute averages
avg_new_init = np.mean(time_init)
avg_new_shift1 = np.mean(time_shift1)
avg_new_shift2 = np.mean(time_shift2)
avg_new_total = np.mean(time_total)

# Print results
print(f"Average initialization time: {avg_new_init:.4f} s")
print(f"Average first shift time: {avg_new_shift1:.4f} s")
print(f"Average second shift time: {avg_new_shift2:.4f} s")
print(f"Average total time per run: {avg_new_total:.4f} s")


Average initialization time: 2.3429 s
Average first shift time: 0.1526 s
Average second shift time: 0.3205 s
Average total time per run: 2.8160 s


In [6]:
# compare avg_main and avg_new
print(f"Difference initialization time: {avg_old_init - avg_new_init:.4f} s")
print(f"Difference first shift time: {avg_old_shift1 - avg_new_shift1:.4f} s")
print(f"Difference second shift time: {avg_old_shift2 - avg_new_shift2:.4f} s")
print(f"Difference total time per run: {avg_old_total - avg_new_total:.4f} s")

print(f"\nshape: {pd_main.data.ts.shape}\nNumber of Runs: {n_runs}\n-> diff = old - new")

Difference initialization time: 0.0895 s
Difference first shift time: 0.7472 s
Difference second shift time: 1.9548 s
Difference total time per run: 2.7915 s

shape: (500, 200, 400)
Number of Runs: 10
-> diff = old - new


## Check if both Version create same Results

In [1]:
from ponds.blobgen_old import GAUSS as GAUSS_old
from ponds.blobgen import GAUSS as GAUSS_new

from ponds.core_old import PONDS as PONDS_old
from ponds.core import PONDS as PONDS_new

from ponds.shiftgen_old import SIGMOID as SIGMOID_old
from ponds.shiftgen import SIGMOID as SIGMOID_new

In [6]:
lat_size = 200
lon_size = 400
time_size = 200

In [7]:
# initialize PONDS object
pd_main = PONDS_old(
    lat_size=lat_size,
    lon_size=lon_size,
    time_size=time_size,
    background_noise=0.01,
    background_trend=0.002,
)

# add random shift
pd_main.add_shift(
    random_seed=1,
    shape_method=GAUSS_old,
    shift_method=SIGMOID_old,
)

# add shift with custom parameters
shift_dict = {
    "shift_steepness": 0.2,
    "shift_magnitude": 0.5,
}
pd_main.add_shift(
    random_seed=4,
    shape_method=GAUSS_old,
    shift_method=SIGMOID_old,
    kwargs=shift_dict,
)

In [8]:
# initialize PONDS object
pd_new = PONDS_new(
    lat_size=lat_size,
    lon_size=lon_size,
    time_size=time_size,
    background_noise=0.01,
    background_trend=0.002,
)

# add random shift
pd_new.add_shift(
    random_seed=1,
    shape_method=GAUSS_new(random_seed=1),
    shift_method=SIGMOID_new(random_seed=1),
)

# add shift with custom parameters
shift_dict = {
    "shift_steepness": 0.2,
    "shift_magnitude": 0.5,
}
pd_new.add_shift(
    random_seed=4,
    shape_method=GAUSS_new(random_seed=4),
    shift_method=SIGMOID_new(random_seed=4,**shift_dict),
    #kwargs=shift_dict,
)

In [9]:
import numpy as np

ts_main = pd_main.data.ts.values
ts_new = pd_new.data.ts.values

print(ts_main.shape)
print(ts_new.shape)

err = []
for t in range(len(ts_main)):
    for i in range(len(ts_main[0])):
        for j in range(len(ts_main[0,0])):
            if ts_main[t,i,j] == ts_new[t,i,j]:
                continue
            else:
                err.append(abs(ts_main[t,i,j] - ts_new[t,i,j]))

err = np.array(err)
print(err.mean())
print(f"{len(err)}/{np.prod(ts_main.shape)} -> {len(err)/np.prod(ts_main.shape)*100} %")

np.array_equal(ts_main,ts_new)

(200, 200, 400)
(200, 200, 400)
0.005099715751865847
415801/16000000 -> 2.5987562499999997 %


False

In [11]:
print(err)

[]
